# TriageAI — Local Deployment with Ollama
### Offline Emergency Triage on Any Laptop

This notebook demonstrates TriageAI running **locally via Ollama** — no cloud, no internet, complete privacy.

| Detail | Value |
|---|---|
| Runtime | Ollama (local inference) |
| Model | Gemma 4 E2B / E4B |
| Prize | Ollama $10K Special Prize |
| Key Feature | Fully offline emergency triage |

In [ ]:
%%capture
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q requests

In [ ]:
import subprocess
import time
import os

# Start Ollama server in background
env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env
)
time.sleep(5)
print("Ollama server started.")

In [ ]:
# Pull Gemma 4 E2B model
!ollama pull gemma4:e2b
print("Gemma 4 E2B pulled successfully.")

## Create TriageAI Modelfile

A custom Modelfile embeds our triage system prompt into the model configuration.

In [ ]:
MODELFILE = """FROM gemma4:e2b

SYSTEM \"\"\"You are TriageAI, an expert emergency triage assistant.
You follow the START (Simple Triage and Rapid Treatment) protocol:
- RED (Immediate): Life-threatening, needs immediate intervention
- YELLOW (Delayed): Serious but can wait briefly
- GREEN (Minor): Walking wounded
- BLACK (Expectant): Beyond current help

For every emergency:
1. Classify the type (medical/fire/flood/earthquake/accident)
2. Assess severity (RED/YELLOW/GREEN/BLACK)
3. Provide step-by-step actions
4. List DO NOT warnings
5. Include emergency number

Be direct, clear, and actionable. Lives depend on your guidance.
Always include a medical disclaimer.\"\"\"

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_predict 512
"""

with open("/tmp/TriageAI.Modelfile", "w") as f:
    f.write(MODELFILE)

!ollama create triageai -f /tmp/TriageAI.Modelfile
print("TriageAI model created in Ollama.")

## Test Cases

In [ ]:
import requests
import json

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

def triage_ollama(scenario: str, model: str = "triageai") -> str:
    """Run a triage query through Ollama."""
    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": scenario,
        "stream": False,
    }, timeout=120)
    return response.json()["response"]

# Test Case 1: Severe Bleeding (English)
print("=" * 60)
print("TEST 1: Severe Arm Laceration (English)")
print("=" * 60)
result = triage_ollama(
    "My friend fell on broken glass and has a deep cut on his forearm. "
    "There's a lot of blood spurting out and he's getting pale. What do I do?"
)
print(result)
print()

In [ ]:
# Test Case 2: Earthquake (Spanish)
print("=" * 60)
print("TEST 2: Earthquake (Spanish)")
print("=" * 60)
result = triage_ollama(
    "Hubo un terremoto fuerte. Mi vecina está atrapada bajo escombros. "
    "Puedo ver su brazo pero no responde. Hay cables eléctricos caídos cerca. ¿Qué hago?"
)
print(result)
print()

In [ ]:
# Test Case 3: Cardiac Emergency (Hindi)
print("=" * 60)
print("TEST 3: Cardiac Emergency (Hindi)")
print("=" * 60)
result = triage_ollama(
    "मेरे पिताजी अचानक सीने में दर्द की शिकायत करते हुए गिर गए हैं। "
    "वे सांस नहीं ले रहे हैं। कृपया मदद करें!"
)
print(result)
print()

In [ ]:
# Latency benchmark
import time

scenarios = [
    "Someone is choking on food and can't breathe.",
    "There's a fire in the kitchen and smoke is filling the room.",
    "A child fell from a tree and can't move his leg.",
]

print("=" * 60)
print("LATENCY BENCHMARK")
print("=" * 60)
for i, s in enumerate(scenarios, 1):
    start = time.time()
    result = triage_ollama(s)
    elapsed = time.time() - start
    word_count = len(result.split())
    print(f"Scenario {i}: {elapsed:.1f}s | {word_count} words | {word_count/elapsed:.0f} words/sec")

## Summary

TriageAI running via **Ollama** demonstrates:
- **Fully offline** emergency triage — no internet needed
- **Multilingual** support (English, Spanish, Hindi, and more)
- **Fast local inference** on consumer hardware
- **Custom Modelfile** with embedded triage system prompt
- **Privacy-first** — no data leaves the device

In a disaster scenario where cell towers are down, this is the difference between having expert guidance and having nothing.

---
*TriageAI — Ollama Special Prize ($10K)*